In [1]:
import numpy as np
import pandas as pd
import glob
from helper import get_data_splits, id_key, get_model
from define_model import create_MultiTaskTouchPoseNet
from my_helpers import *
from itertools import zip_longest
import pprint as p

2023-07-17 15:01:36.631781: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-07-17 15:01:37.500414: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
# Below identifiers correspond to evaluation protocol 1, 2 and 3 respectively
# identifiers - "TouchPose_crossSession", "TouchPose_crossSubjects", "TouchPose_crossGestures"
identifier = "TouchPose_crossGestures"
data_path = "../data/eth_data/"
model_path = "../models/"
new_data = "../data/27Arish"

# Information from Report

The valid skeleton data we collected from leap motion controller covered 147 trails,
with an average number of frames per trail of 1316. The frame rate of Leap motion data is
30 fps.

The mediapipe skeleton data we collected from hand writing on Wacom tablet covered
200 trails. The average number of frames per trial is 1540. The working fps of RGB camera
is 30 fps.

The capacity image and stylus data were recorded from Wacom tablet. The capacitive
image has the resolution 72*42. We collected 132 trails of capacity image and 201 trails
of pen data in total. The average number of frames of capacitive data per trial is 13203
frames, and the number of frames of pen data per trial is 12262. The recording rate is
about 80 fps.

In [113]:
cap_img = read_cap_img(new_data+"/Arish_1_UP_100_rawdata.csv")
skeleton = read_skeleton(new_data+"/userarish_2023-04-03_10-10-07_11956_handleap.csv")

In [114]:
skeleton.Timestamp = skeleton.Timestamp*1000
skeleton.Timestamp = skeleton.Timestamp.astype('int64')
cap_img = cap_img.rename(columns={'TimeStamps': 'Timestamp'})
cap_img.head()

,Timestamp,MergeTimeStamps,ScanSizeX,ScanSizeY,img
0,1680508053473,1680508053470,72,42,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1,1680508053489,1680508053485,72,42,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
2,1680508053506,1680508053505,72,42,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
3,1680508053519,1680508053515,72,42,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
4,1680508053532,1680508053530,72,42,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


In [115]:
skeleton.Timestamp.diff().mean()

114.01062317022166

In [116]:
cap_img.Timestamp.diff().mean()

13.382334284808119

It looks like MergeTimeStamps is the biggest smaller value mutiple of 5 with respect to TimeStamps.
We do the same for skeleton.

In [118]:
skeleton['MergeTimeStamps']=(skeleton.Timestamp// 5) * 5
skeleton.head()

,Timestamp,wrist_x,wrist_y,wrist_z,Thumb_0_x,Thumb_0_y,Thumb_0_z,Thumb_1_x,Thumb_1_y,Thumb_1_z,...,Pinky_1_x,Pinky_1_y,Pinky_1_z,Pinky_2_x,Pinky_2_y,Pinky_2_z,Pinky_3_x,Pinky_3_y,Pinky_3_z,MergeTimeStamps
0,1680508023335,-139.203323,274.228577,49.082916,-114.933983,253.419250,30.097525,-114.933983,253.419250,30.097525,...,-101.600784,312.727509,-1.378431,-79.235779,316.579315,-9.686751,-66.095505,310.570587,-4.683544,1680508023335
1,1680508023361,-140.807953,275.914978,48.604588,-116.763191,254.998688,29.973202,-116.763191,254.998688,29.973202,...,-103.461205,313.773743,-1.610328,-81.087822,317.742035,-10.036551,-67.429924,312.277679,-5.582808,1680508023360
2,1680508023370,-141.657455,276.928864,48.438431,-117.944008,255.992447,29.780630,-117.944008,255.992447,29.780630,...,-104.594521,314.625885,-1.542735,-82.366440,318.522095,-10.068500,-68.662849,313.191895,-6.016860,1680508023370
3,1680508023382,-142.424988,277.627655,48.370846,-118.904243,256.690155,29.705805,-118.904243,256.690155,29.705805,...,-105.538300,315.195709,-1.483135,-83.491470,319.006104,-10.107798,-69.866707,313.695801,-6.286170,1680508023380
4,1680508023404,-143.209824,278.107635,48.447624,-119.784531,257.230042,29.819784,-119.784531,257.230042,29.819784,...,-106.508232,315.579651,-1.366892,-84.573372,319.315247,-10.042198,-71.030754,313.933380,-6.230742,1680508023400


In [119]:
print(skeleton.iloc[0].Timestamp < cap_img.iloc[0].Timestamp)
print(skeleton.iloc[-1].Timestamp > cap_img.iloc[-1].Timestamp)

True
True


Get overlapping between skeleton and capactive images

In [120]:
skeleton=skeleton[((skeleton.Timestamp >= cap_img.Timestamp.iloc[0]) & (skeleton.Timestamp <= cap_img.Timestamp.iloc[-1]))].reset_index(drop=True)
len(skeleton)

3494

In [121]:
len(cap_img)

6307

Hence we want to get 3494 observations, we will map skeleton to average of capactivie images.

using np.digitize

In [134]:
cap_img

,Timestamp,MergeTimeStamps,ScanSizeX,ScanSizeY,img,where_to
0,1680508053473,1680508053470,72,42,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0
1,1680508053489,1680508053485,72,42,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",1
2,1680508053506,1680508053505,72,42,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",2
3,1680508053519,1680508053515,72,42,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",3
4,1680508053532,1680508053530,72,42,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",3
...,...,...,...,...,...,...
6302,1680508137815,1680508137815,72,42,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",3494
6303,1680508137827,1680508137825,72,42,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",3494
6304,1680508137838,1680508137835,72,42,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",3494
6305,1680508137851,1680508137850,72,42,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",3494


In [138]:
skeleton.Timestamp.diff().sort_values(ascending=False)  

3378    7802.0
567     7326.0
10        74.0
2447      65.0
2859      59.0
         ...  
616        7.0
449        7.0
1567       6.0
591        6.0
0          NaN
Name: Timestamp, Length: 3494, dtype: float64

In [135]:
cap_img.Timestamp.diff().mean() 


16.621823911758913

In [133]:
skeleton

,Timestamp,wrist_x,wrist_y,wrist_z,Thumb_0_x,Thumb_0_y,Thumb_0_z,Thumb_1_x,Thumb_1_y,Thumb_1_z,...,Pinky_1_x,Pinky_1_y,Pinky_1_z,Pinky_2_x,Pinky_2_y,Pinky_2_z,Pinky_3_x,Pinky_3_y,Pinky_3_z,MergeTimeStamps
0,1680508053484,-256.720947,295.964569,92.471275,-233.598572,279.741333,76.260818,-233.598572,279.741333,76.260818,...,-247.508896,316.530426,35.776749,-229.505341,322.601776,28.719210,-220.962616,320.603851,40.602127,1680508053480
1,1680508053505,-256.187317,296.331665,92.525307,-233.457581,279.602448,76.259453,-233.457581,279.602448,76.259453,...,-246.059570,317.245453,36.092812,-227.599884,323.007538,29.602783,-219.499023,320.498779,41.970451,1680508053505
2,1680508053517,-256.325500,296.753510,92.741585,-233.637848,279.903046,76.547279,-233.637848,279.903046,76.547279,...,-245.604721,317.915497,36.570469,-226.914276,323.522461,30.488916,-219.062225,320.859924,42.970566,1680508053515
3,1680508053544,-256.242737,296.946594,92.780807,-233.588394,280.024902,76.661621,-233.588394,280.024902,76.661621,...,-245.080582,318.226929,36.725681,-226.225296,323.759125,30.841139,-218.462402,321.017731,43.464756,1680508053540
4,1680508053554,-256.411224,296.899170,92.889946,-233.705902,279.967712,76.914787,-233.705902,279.967712,76.914787,...,-244.962708,318.267700,36.925682,-226.092133,323.773102,31.171854,-218.389832,321.024933,43.853363,1680508053550
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3489,1680508136121,-210.610245,259.248352,156.621613,-183.260025,243.383163,148.467300,-183.260025,243.383163,148.467300,...,-195.706360,260.662476,94.836884,-178.328156,266.595886,85.906731,-169.940613,269.540680,95.457108,1680508136120
3490,1680508136135,-208.178207,263.949066,162.127350,-180.091797,249.819656,154.950119,-180.091797,249.819656,154.950119,...,-192.657104,260.753143,99.456261,-175.238068,267.159058,90.506508,-167.606003,271.681793,99.837105,1680508136135
3491,1680508136158,-206.146286,267.992401,166.687958,-177.379303,255.752472,159.677109,-177.379303,255.752472,159.677109,...,-189.671158,262.540619,103.975647,-172.346313,269.205566,95.603378,-165.655746,274.684692,105.458389,1680508136155
3492,1680508136172,-201.941086,273.434235,169.972946,-173.179031,262.728546,162.759354,-173.179031,262.728546,162.759354,...,-184.922318,263.058441,106.845490,-168.159882,269.801361,98.883354,-163.001907,276.367432,109.180008,1680508136170


In [132]:
bins = skeleton.Timestamp
x = cap_img.Timestamp
cap_img['where_to'] = np.digitize(x, bins)
cap_img.where_to.unique()

array([   0,    1,    2, ..., 3492, 3493, 3494])

In [123]:
ranking =skeleton.Timestamp.diff().sort_values(ascending=False)
ranking=ranking[ranking > 80]
ranking

3378    7802.0
567     7326.0
Name: Timestamp, dtype: float64

In [124]:
cap_img=cap_img[~cap_img.where_to.isin(ranking.index)]
res=cap_img.groupby('where_to')['img'].agg(lambda x : np.mean(np.array(x)))

In [130]:
cap_img.where_to.value_counts()

3494    144
10        6
2859      5
2447      4
992       4
       ... 
1930      1
1929      1
627       1
628       1
0         1
Name: where_to, Length: 3398, dtype: int64

In [125]:
len(res)

3398

In [128]:
skeleton

,Timestamp,wrist_x,wrist_y,wrist_z,Thumb_0_x,Thumb_0_y,Thumb_0_z,Thumb_1_x,Thumb_1_y,Thumb_1_z,...,Pinky_1_x,Pinky_1_y,Pinky_1_z,Pinky_2_x,Pinky_2_y,Pinky_2_z,Pinky_3_x,Pinky_3_y,Pinky_3_z,MergeTimeStamps
0,1680508053484,-256.720947,295.964569,92.471275,-233.598572,279.741333,76.260818,-233.598572,279.741333,76.260818,...,-247.508896,316.530426,35.776749,-229.505341,322.601776,28.719210,-220.962616,320.603851,40.602127,1680508053480
1,1680508053505,-256.187317,296.331665,92.525307,-233.457581,279.602448,76.259453,-233.457581,279.602448,76.259453,...,-246.059570,317.245453,36.092812,-227.599884,323.007538,29.602783,-219.499023,320.498779,41.970451,1680508053505
2,1680508053517,-256.325500,296.753510,92.741585,-233.637848,279.903046,76.547279,-233.637848,279.903046,76.547279,...,-245.604721,317.915497,36.570469,-226.914276,323.522461,30.488916,-219.062225,320.859924,42.970566,1680508053515
3,1680508053544,-256.242737,296.946594,92.780807,-233.588394,280.024902,76.661621,-233.588394,280.024902,76.661621,...,-245.080582,318.226929,36.725681,-226.225296,323.759125,30.841139,-218.462402,321.017731,43.464756,1680508053540
4,1680508053554,-256.411224,296.899170,92.889946,-233.705902,279.967712,76.914787,-233.705902,279.967712,76.914787,...,-244.962708,318.267700,36.925682,-226.092133,323.773102,31.171854,-218.389832,321.024933,43.853363,1680508053550
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3489,1680508136121,-210.610245,259.248352,156.621613,-183.260025,243.383163,148.467300,-183.260025,243.383163,148.467300,...,-195.706360,260.662476,94.836884,-178.328156,266.595886,85.906731,-169.940613,269.540680,95.457108,1680508136120
3490,1680508136135,-208.178207,263.949066,162.127350,-180.091797,249.819656,154.950119,-180.091797,249.819656,154.950119,...,-192.657104,260.753143,99.456261,-175.238068,267.159058,90.506508,-167.606003,271.681793,99.837105,1680508136135
3491,1680508136158,-206.146286,267.992401,166.687958,-177.379303,255.752472,159.677109,-177.379303,255.752472,159.677109,...,-189.671158,262.540619,103.975647,-172.346313,269.205566,95.603378,-165.655746,274.684692,105.458389,1680508136155
3492,1680508136172,-201.941086,273.434235,169.972946,-173.179031,262.728546,162.759354,-173.179031,262.728546,162.759354,...,-184.922318,263.058441,106.845490,-168.159882,269.801361,98.883354,-163.001907,276.367432,109.180008,1680508136170


In [126]:
skeleton=skeleton.loc[res.index]
len(skeleton)

KeyError: '[3494] not in index'

In [90]:
skeleton.loc[ = res

ValueError: Length of values (6307) does not match length of index (3494)

use np.digitize to make bins and average frames
or scipy binned statistic


cap img 6k
3k sekeleton

check paper for how they preprocess skeleton

In [14]:
skeleton.Timestamp

0        1680508023335
1        1680508023361
2        1680508023370
3        1680508023382
4        1680508023404
             ...      
11951    1680509386221
11952    1680509386248
11953    1680509386272
11954    1680509386300
11955    1680509386332
Name: Timestamp, Length: 11956, dtype: int64

In [15]:
cap_img.Timestamp

0       1680508053473
1       1680508053489
2       1680508053506
3       1680508053519
4       1680508053532
            ...      
6302    1680508137815
6303    1680508137827
6304    1680508137838
6305    1680508137851
6306    1680508137862
Name: Timestamp, Length: 6307, dtype: int64

In [10]:
print(len(cap_img), len(skeleton))
print(len(cap_img.Timestamp.unique()), len(skeleton.Timestamp.unique()))


6307 11956
6307 11956


In [11]:
both=pd.merge(skeleton, cap_img[['img','MergeTimeStamps','Timestamp']], on='MergeTimeStamps', how='inner', suffixes=('_skeleton', '_cap_img'))
print(len(both))
both.columns

1255


Index(['Timestamp_skeleton', 'wrist_x', 'wrist_y', 'wrist_z', 'Thumb_0_x',
       'Thumb_0_y', 'Thumb_0_z', 'Thumb_1_x', 'Thumb_1_y', 'Thumb_1_z',
       'Thumb_2_x', 'Thumb_2_y', 'Thumb_2_z', 'Thumb_3_x', 'Thumb_3_y',
       'Thumb_3_z', 'Index_0_x', 'Index_0_y', 'Index_0_z', 'Index_1_x',
       'Index_1_y', 'Index_1_z', 'Index_2_x', 'Index_2_y', 'Index_2_z',
       'Index_3_x', 'Index_3_y', 'Index_3_z', 'Middle_0_x', 'Middle_0_y',
       'Middle_0_z', 'Middle_1_x', 'Middle_1_y', 'Middle_1_z', 'Middle_2_x',
       'Middle_2_y', 'Middle_2_z', 'Middle_3_x', 'Middle_3_y', 'Middle_3_z',
       'Ring_0_x', 'Ring_0_y', 'Ring_0_z', 'Ring_1_x', 'Ring_1_y', 'Ring_1_z',
       'Ring_2_x', 'Ring_2_y', 'Ring_2_z', 'Ring_3_x', 'Ring_3_y', 'Ring_3_z',
       'Pinky_0_x', 'Pinky_0_y', 'Pinky_0_z', 'Pinky_1_x', 'Pinky_1_y',
       'Pinky_1_z', 'Pinky_2_x', 'Pinky_2_y', 'Pinky_2_z', 'Pinky_3_x',
       'Pinky_3_y', 'Pinky_3_z', 'MergeTimeStamps', 'img',
       'Timestamp_cap_img'],
      dtype='object

In [12]:
data = both.copy()
hand_cords = data.drop(["img","Timestamp_cap_img","Timestamp_skeleton", 'MergeTimeStamps'],axis=1)
data['skeleton'] = hand_cords.values.tolist()
data = data.drop(hand_cords.columns, axis=1)
data.skeleton = data.skeleton.map(lambda x: np.array(x).reshape(21,3))
data.to_pickle('../data/gen_data/Arish_1_UP_100_.pkl')

In [16]:
np.mean(np.absolute(data.Timestamp_skeleton - data.Timestamp_cap_img))

1.5784860557768925